## Conditional Workflows ##

**Non LLM Workflow**

In [6]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 1. State Definition
class QuadState(TypedDict):
    a: int
    b: int
    c: int
    equation: str
    discriminant: float
    result: str

# 2. Node Functions
def show_equation(state: QuadState):
    equation = f'{state["a"]}x^2 + {state["b"]}x + {state["c"]}'
    return {"equation": equation}

def calculate_discriminant(state: QuadState):
    discriminant = state["b"]**2 - (4 * state["a"] * state["c"])
    return {"discriminant": discriminant}

def real_roots(state: QuadState):
    root1 = (-state["b"] + state["discriminant"]**0.5) / (2 * state["a"])
    root2 = (-state["b"] - state["discriminant"]**0.5) / (2 * state["a"])
    result = f'The roots are {root1} and {root2}'
    return {'result': result}

def repeated_roots(state: QuadState):
    root = (-state["b"]) / (2 * state["a"])
    result = f'Only repeating root is {root}'
    return {'result': result}

def no_real_roots(state: QuadState):
    result = 'No real roots'
    return {'result': result}

# 3. Conditional Router
def check_condition(state: QuadState) -> Literal["real_roots", "repeated_roots", "no_real_roots"]:
    if state['discriminant'] > 0:
        return "real_roots"
    elif state['discriminant'] == 0:
        return "repeated_roots"
    else:
        return "no_real_roots"

# 4. Build Graph
graph = StateGraph(QuadState)

# Add Nodes
graph.add_node("show_equation", show_equation)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("real_roots", real_roots)
graph.add_node("repeated_roots", repeated_roots)
graph.add_node("no_real_roots", no_real_roots)

# Add Edges
graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")

# Add Conditional Edges from Discriminant Node
graph.add_conditional_edges("calculate_discriminant", check_condition)

# Connect calculation terminals to END
graph.add_edge("real_roots", END)
graph.add_edge("repeated_roots", END)
graph.add_edge("no_real_roots", END)

# 5. Compile and Invoke
workflow = graph.compile()

initial_state = {"a": 1, "b": -5, "c": 6}

result = workflow.invoke(initial_state)

print("Equation:", result["equation"])
print("Discriminant:", result["discriminant"])
print("Result:", result["result"])

Equation: 1x^2 + -5x + 6
Discriminant: 1
Result: The roots are 3.0 and 2.0


**LLM Workflow**


In [1]:
import os
from typing import Literal, TypedDict, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

# ================================
# 1. SCHEMAS & STATE DEFINITION
# ================================

class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description="The category of the issue")
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description="The emotional tone")
    urgency: Literal["low", "medium", "high"] = Field(description="How urgent or critical the issue is")

class ReviewState(TypedDict):
    review: str
    sentiment: Optional[str]
    diagnosis: Optional[dict]
    response: Optional[str]

# ================================
# 2. MODEL INITIALIZATION
# ================================

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
structured_model1 = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

# ================================
# 3. NODE FUNCTIONS & ROUTERS
# ================================

def find_sentiment(state: ReviewState):
    prompt = f"Analyze the sentiment of this review:\n\n\"{state['review']}\""
    response = structured_model1.invoke(prompt)
    return {'sentiment': response.sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:

"{state['review']}"

Also, kindly ask the user to leave feedback on our website."""

    response = model.invoke(prompt).content
    return {'response': response}

def run_diagnosis(state: ReviewState):
    prompt = f"""Diagnose this negative review:\n\n"{state['review']}"
Return issue_type, tone, and urgency."""

    response = structured_model2.invoke(prompt)
    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message."""

    response = model.invoke(prompt).content
    return {'response': response}

# ================================
# 4. BUILD & COMPILE GRAPH
# ================================

graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

# ================================
# 5. EXECUTION EXAMPLES
# ================================

if __name__ == "__main__":
    # Test 1: Negative Review Pipeline
    print("--- Testing Negative Review ---")
    negative_review_input = {
        "review": "The app keeps crashing every time I try to checkout! Fix this ASAP, I lost my order."
    }
    result_neg = workflow.invoke(negative_review_input)
    print("Sentiment:", result_neg.get("sentiment"))
    print("Diagnosis:", result_neg.get("diagnosis"))
    print("Generated Response:\n", result_neg.get("response"))

    print("\n" + "=" * 50 + "\n")

    # Test 2: Positive Review Pipeline
    print("--- Testing Positive Review ---")
    positive_review_input = {
        "review": "I absolutely love this app! It made ordering groceries so fast and effortless."
    }
    result_pos = workflow.invoke(positive_review_input)
    print("Sentiment:", result_pos.get("sentiment"))
    print("Generated Response:\n", result_pos.get("response"))